In [1]:
# ============================================================
# CELL 1 (per official PaddleOCR docs): PaddlePaddle GPU + PaddleOCR
# ============================================================
!pip uninstall -y paddlepaddle paddlepaddle-gpu paddleocr paddlex -q
!python -m pip install -q paddlepaddle==3.2.0 -i https://www.paddlepaddle.org.cn/packages/stable/cpu/
!python -m pip install -q paddleocr

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 189.0/189.0 MB 6.0 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 3.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.7/80.7 kB 1.2 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.8/146.8 kB 2.4 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 14.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 767.5/767.5 kB 31.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 MB 23.0 MB/s eta 0:00:0000:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.2/67.2 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.0/6.0 MB 70.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 81.3 MB/s et

In [2]:
# ============================================================
# CELL 1b: Verify PaddlePaddle GPU install (per official docs)
# ============================================================
import paddle
print(paddle.__version__)
paddle.utils.run_check()

/usr/local/lib/python3.12/dist-packages/paddle/utils/cpp_extension/extension_utils.py:718: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)


3.2.0
Running verify PaddlePaddle program ... 
PaddlePaddle works well on 1 CPU.
PaddlePaddle is installed successfully! Let's start deep learning with PaddlePaddle now.


/usr/local/lib/python3.12/dist-packages/paddle/pir/math_op_patch.py:219: UserWarning: Value do not have 'place' interface for pir graph mode, try not to use it. None will be returned.
  warnings.warn(
I0826 18:24:21.243816    58 pir_interpreter.cc:1524] New Executor is Running ...
I0826 18:24:21.244238    58 pir_interpreter.cc:1547] pir interpreter is running by multi-thread mode ...


In [3]:
# ============================================================
# CELL 2: Load Pretrained PaddleOCR small
# ============================================================
from paddleocr import PaddleOCR
import time

ocr = PaddleOCR(
    use_doc_orientation_classify=False,
    use_doc_unwarping=False,
    use_textline_orientation=False,
    device='cpu',
    text_detection_model_name="PP-OCRv6_small_det",
    text_recognition_model_name="PP-OCRv6_small_rec",
)
print("PaddleOCR loaded")

import paddle
print("CUDA available:", paddle.device.is_compiled_with_cuda())

Creating model: ('PP-OCRv6_small_det', None, None)
Checking connectivity to the model hosters, this may take a while. To bypass this check, set `PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK` to `True`.
Using official model (PP-OCRv6_small_det), the model files will be automatically downloaded and saved in `/root/.paddlex/official_models/PP-OCRv6_small_det`.


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Creating model: ('PP-OCRv6_small_rec', None, None)
Using official model (PP-OCRv6_small_rec), the model files will be automatically downloaded and saved in `/root/.paddlex/official_models/PP-OCRv6_small_rec`.


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

PaddleOCR loaded
CUDA available: False


In [10]:
import numpy as np

def estimate_skew_angle(polys):
    angles = []
    for poly in polys:
        poly = np.array(poly)
        top_left, top_right = poly[0], poly[1]
        dx = top_right[0] - top_left[0]
        dy = top_right[1] - top_left[1]
        if dx != 0:
            angles.append(np.arctan2(dy, dx))
    return np.median(angles)

def cluster_rows_deskewed(texts, scores, boxes, polys, y_tol_ratio=0.3):
    angle = estimate_skew_angle(polys)
    slope = np.tan(angle)  # dy per dx

    items = []
    for text, score, box in zip(texts, scores, boxes):
        x1, y1, x2, y2 = box
        x_center = (x1 + x2) / 2
        y_center = (y1 + y2) / 2
        y_corrected = y_center - slope * x_center  # remove skew-induced drift
        height = y2 - y1
        items.append({"text": text, "score": score, "x": x1, "y": y_corrected, "h": height})

    items.sort(key=lambda i: i["y"])

    rows = [[items[0]]]
    for item in items[1:]:
        avg_h = sum(i["h"] for i in rows[-1]) / len(rows[-1])
        if abs(item["y"] - rows[-1][-1]["y"]) <= avg_h * y_tol_ratio:
            rows[-1].append(item)
        else:
            rows.append([item])

    for row in rows:
        row.sort(key=lambda i: i["x"])
    return rows

In [11]:
# CELL 3 (FIXED): Baseline Test — Full Receipt Photos 

import numpy as np

def cluster_rows(texts, scores, boxes, y_tol_ratio=0.5):
    items = []
    for text, score, box in zip(texts, scores, boxes):
        x1, y1, x2, y2 = box
        y_center = (y1 + y2) / 2
        height = y2 - y1
        items.append({"text": text, "score": score, "x": x1, "y": y_center, "h": height})

    items.sort(key=lambda i: i["y"])

    rows = []
    current_row = [items[0]]
    for item in items[1:]:
        avg_h = sum(i["h"] for i in current_row) / len(current_row)
        if abs(item["y"] - current_row[-1]["y"]) <= avg_h * y_tol_ratio:
            current_row.append(item)
        else:
            rows.append(current_row)
            current_row = [item]
    rows.append(current_row)

    for row in rows:
        row.sort(key=lambda i: i["x"])

    return rows

for path in image_paths:
    result = ocr.predict(str(path))
    res = result[0]
    # rows = cluster_rows(res["rec_texts"], res["rec_scores"], res["rec_boxes"], y_tol_ratio=0.3)
    rows = cluster_rows_deskewed(res["rec_texts"], res["rec_scores"], res["rec_boxes"], res["rec_polys"])
    print(f"\n=== {path.name} ===")
    for row in rows:
        line = " | ".join(f"[{i['score']:.2f}] {i['text']}" for i in row)
        print(" ", line)


=== 1.jpg ===
  [0.98] FBR Invoice #: 139873231027184240102
  [0.98] -KHI - SUP - TS TOWER-
  [0.99] GST # 12-02-9999-124-64
  [0.98] Transaction No.: 232200081878
  [1.00] Transaction Date: | [0.96] Oct 27, 2023 6:42 PM
  [0.96] User: 57486-Sunil Jain
  [1.00] POS:TS-SAL-020-TS-SAL-020
  [0.98] Customer: 0
  [0.91] CNIC: 0
  [0.99] Duplicate Receipt
  [1.00] Product Description
  [1.00] Quantity | [1.00] Price | [1.00] Discount | [1.00] Total
  [1.00] Sales Items
  [0.98] Supravit-M Tablet 10's
  [1.00] 6.00 | [0.98] 1137.45 | [1.00] 272.99 | [0.99] Rs2,001.91
  [0.99] FBR POS Charges
  [0.98] 1.00 | [1.00] 1.00 | [1.00] 0.00 | [1.00] Rs1.00
  [0.98] Core C Sachet 1's 1000mg
  [1.00] 60.00 | [0.99] 900.00 | [1.00] 216.00 | [0.99] Rs1,584.00
  [0.99] Total Items/Quantity | [1.00] 3/67.00
  [1.00] Discount | [1.00] Rs488.99
  [1.00] Rounding | [1.00] Rs0.00
  [1.00] Invoice Value | [0.95] R53,586.91
  [0.98] Sale Tax Breakup
  [0.96] Ex1. Amt
  [1.00] MRP | [1.00] Rs0.00 | [0.92] Rs0.0